In [3]:
import glob
import re
import numpy as np
import pandas as pd

def find_file(filename):
    matches = glob.glob(
        f"/kaggle/input/**/{filename}",
        recursive=True
    )
    if not matches:
        raise FileNotFoundError(
            f"Nu am gasit {filename} in Kaggle Input!"
        )
    return matches[0]

train = pd.read_csv(find_file("train_context.csv"))
test = pd.read_csv(find_file("test_context.csv"))

print("Train:", train.shape)
print("Test:", test.shape)

print("\nColoanele disponibile:")
print(train.columns.tolist())

display(train.head(3))


Train: (3278, 15)
Test: (296, 12)

Coloanele disponibile:
['datapointID', 'title', 'date', 'president', 'url', 'context', 'question_order', 'interview_question', 'interview_answer', 'clarity_label', 'evasion_label', 'fold', 'turn_gap', 'prev_question', 'prev_answer']


,datapointID,title,date,president,url,context,question_order,interview_question,interview_answer,clarity_label,evasion_label,fold,turn_gap,prev_question,prev_answer
0,1,CONFERINȚA DE PRESĂ COMUNĂ a Președintelui Rom...,2001-02-09,Ion Iliescu,https://arhiva.presidency.ro/html/date0/Confer...,Press Conference,1,Pentru domnul Președinte Iliescu. România va f...,Interdicții din partea cui?,Clear Non-Reply,Clarification,4,-1.0,NaN,NaN
1,2,CONFERINȚA DE PRESĂ COMUNĂ a Președintelui Rom...,2001-02-09,Ion Iliescu,https://arhiva.presidency.ro/html/date0/Confer...,Press Conference,3,Am avut onoarea să particip la întâlnirea de l...,Am evocat chiar astăzi o asemenea perspectivă ...,Clear Reply,Explicit,0,2.0,NaN,NaN
2,3,CONFERINȚA DE PRESĂ COMUNĂ a Președintelui Rom...,2001-02-09,Ion Iliescu,https://arhiva.presidency.ro/html/date0/Confer...,Press Conference,4,"Domnule Președinte Iliescu, fiindcă, tangenția...",Domnul ministru Geoană a fost de curând la Mos...,Ambivalent,Implicit,4,1.0,Am avut onoarea să particip la întâlnirea de l...,Am evocat chiar astăzi o asemenea perspectivă ...


In [4]:

EVASION_CLASSES = [
    'Explicit',
    'Implicit',
    'General',
    'Partial/half-answer',
    'Dodging',
    'Deflection',
    'Declining to answer',
    'Claims ignorance',
    'Clarification'
]

required_train = [
    "datapointID", "fold", "evasion_label",
    "interview_question", "interview_answer",
    "president", "context", "question_order",
    "turn_gap"
]

required_test = [
    c for c in required_train
    if c not in ["fold", "evasion_label"]
]

assert all(c in train.columns for c in required_train)
assert all(c in test.columns for c in required_test)

assert len(train) == 3278
assert len(test) == 296

assert train["datapointID"].is_unique
assert test["datapointID"].is_unique

assert set(train["fold"]) == {0, 1, 2, 3, 4}
assert set(train["evasion_label"]) == set(EVASION_CLASSES)

print("Datele sunt compatibile cu structura V4!")


Datele sunt compatibile cu structura V4!


In [5]:

def extract_features(df):
    features = pd.DataFrame(index=df.index)

    question = df["interview_question"].fillna("").astype(str)
    answer = df["interview_answer"].fillna("").astype(str)

    q_lower = question.str.lower()
    a_lower = answer.str.lower()

    # 1. Lungimea textelor
    features["question_words"] = question.str.split().str.len()
    features["answer_words"] = answer.str.split().str.len()

    features["question_chars"] = question.str.len()
    features["answer_chars"] = answer.str.len()

    features["length_ratio"] = (
        features["answer_words"] /
        (features["question_words"] + 1)
    )

    # 2. Punctuatie si structura
    features["answer_question_marks"] = answer.str.count(r"\?")
    features["answer_exclamations"] = answer.str.count("!")
    features["answer_commas"] = answer.str.count(",")

    features["answer_sentences"] = (
        answer.str.count(r"[.!?]") + 1
    )

    # 3. Tipare lingvistice
    
    features["claims_ignorance"] = (
        a_lower.str.contains(
            r"\bnu (?:știu|stiu|cunosc)\b|"
            r"\bnu (?:am|dețin|detin) (?:aceste )?(?:date|informații|informatii)\b",
            regex=True
        )
    ).astype(int)


    features["refuses_answer"] = (
        a_lower.str.contains(
            r"nu comentez|nu voi comenta|"
            r"nu mă pronunț|nu ma pronunt|"
            r"nu pot răspunde|nu pot raspunde",
            regex=True
        )
    ).astype(int)

    features["asks_clarification"] = (
        a_lower.str.contains(
            r"vă referiți|va referiti|"
            r"care dintre|ce anume|"
            r"din partea cui",
            regex=True
        )
    ).astype(int)

    # 4. Suprapunerea lexicala
    def lexical_overlap(q, a):
        q_words = set(re.findall(r"\w+", q.lower()))
        a_words = set(re.findall(r"\w+", a.lower()))

        if not q_words:
            return 0.0

        return len(q_words & a_words) / len(q_words)

    features["lexical_overlap"] = [
        lexical_overlap(q, a)
        for q, a in zip(question, answer)
    ]

    # 5. Context conversational
    features["turn_gap"] = df["turn_gap"].fillna(-1)
    features["has_previous_turn"] = (
        features["turn_gap"] == 1
    ).astype(int)

    features["question_order"] = df["question_order"].fillna(0)

    # 6. Metadate categorice
    features["president"] = (
        df["president"].fillna("Unknown").astype("category")
    )

    features["context"] = (
        df["context"].fillna("Unknown").astype("category")
    )

    return features


X_train = extract_features(train)
X_test = extract_features(test)

print("Train features:", X_train.shape)
print("Test features:", X_test.shape)

display(X_train.head())


Train features: (3278, 18)
Test features: (296, 18)


,question_words,answer_words,question_chars,answer_chars,length_ratio,answer_question_marks,answer_exclamations,answer_commas,answer_sentences,claims_ignorance,refuses_answer,asks_clarification,lexical_overlap,turn_gap,has_previous_turn,question_order,president,context
0,22,4,137,27,0.173913,1,0,0,2,0,0,1,0.047619,-1.0,0,1,Ion Iliescu,Press Conference
1,36,78,211,474,2.108108,0,0,3,3,0,0,0,0.352941,2.0,0,3,Ion Iliescu,Press Conference
2,35,80,279,509,2.222222,0,0,10,6,0,0,0,0.151515,1.0,1,4,Ion Iliescu,Press Conference
3,41,8,248,37,0.190476,0,0,0,3,0,0,0,0.027027,3.0,0,7,Ion Iliescu,Press Conference
4,26,35,187,193,1.296296,0,0,4,3,1,0,0,0.240000,3.0,0,10,Ion Iliescu,Press Conference


In [6]:

print("Numar de caracteristici:", X_train.shape[1])

print("\nValori lipsa in train:")
print(X_train.isnull().sum())

print("\nValori lipsa in test:")
print(X_test.isnull().sum())

print("\nDistributia indicatorilor lingvistici:")

for col in [
    "claims_ignorance",
    "refuses_answer",
    "asks_clarification"
]:
    print(col, ":", X_train[col].sum())

assert len(X_train) == len(train)
assert len(X_test) == len(test)

print("\nVerificari finalizate!")


Numar de caracteristici: 18

Valori lipsa in train:
question_words           0
answer_words             0
question_chars           0
answer_chars             0
length_ratio             0
answer_question_marks    0
answer_exclamations      0
answer_commas            0
answer_sentences         0
claims_ignorance         0
refuses_answer           0
asks_clarification       0
lexical_overlap          0
turn_gap                 0
has_previous_turn        0
question_order           0
president                0
context                  0
dtype: int64

Valori lipsa in test:
question_words           0
answer_words             0
question_chars           0
answer_chars             0
length_ratio             0
answer_question_marks    0
answer_exclamations      0
answer_commas            0
answer_sentences         0
claims_ignorance         0
refuses_answer           0
asks_clarification       0
lexical_overlap          0
turn_gap                 0
has_previous_turn        0
question_order       

In [7]:

import lightgbm as lgb
from sklearn.metrics import f1_score

label2id = {
    c: i for i, c in enumerate(EVASION_CLASSES)
}

y_train = train["evasion_label"].map(label2id).to_numpy()

assert not pd.isna(y_train).any()

for col in ["president", "context"]:
    categories = pd.Index(
        pd.concat([
            X_train[col].astype(str),
            X_test[col].astype(str)
        ]).unique()
    )

    X_train[col] = pd.Categorical(
        X_train[col], categories=categories
    )

    X_test[col] = pd.Categorical(
        X_test[col], categories=categories
    )

assert X_train.index.equals(train.index)
assert X_test.index.equals(test.index)
assert set(train["fold"]) == {0, 1, 2, 3, 4}

print("LightGBM:", lgb.__version__)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Clase:", len(EVASION_CLASSES))
print("Datele sunt pregatite pentru antrenare!")


LightGBM: 4.6.0
X_train: (3278, 18)
X_test: (296, 18)
Clase: 9
Datele sunt pregatite pentru antrenare!


In [8]:

oof_lgb = np.zeros((len(train), 9))
test_lgb = np.zeros((len(test), 9))

for fold in range(5):
    print(f"\n=== FOLD {fold + 1}/5 ===")

    tr_mask = train["fold"] != fold
    va_mask = train["fold"] == fold

    model = lgb.LGBMClassifier(
        objective="multiclass",
        num_class=9,
        n_estimators=300,
        learning_rate=0.03,
        num_leaves=15,
        max_depth=5,
        min_child_samples=20,
        class_weight="balanced",
        reg_lambda=2.0,
        verbosity=-1,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train.loc[tr_mask],
        y_train[tr_mask],
        categorical_feature=["president", "context"]
    )

    val_probs = model.predict_proba(
        X_train.loc[va_mask]
    )

    test_probs_fold = model.predict_proba(X_test)

    assert np.array_equal(model.classes_, np.arange(9))

    oof_lgb[va_mask.to_numpy()] = val_probs
    test_lgb += test_probs_fold / 5.0

    val_pred = val_probs.argmax(axis=1)

    fold_f1 = f1_score(
        y_train[va_mask],
        val_pred,
        labels=np.arange(9),
        average="macro",
        zero_division=0
    )

    print(f"Macro-F1 fold {fold + 1}: {fold_f1:.4f}")

oof_pred = oof_lgb.argmax(axis=1)

final_f1 = f1_score(
    y_train,
    oof_pred,
    labels=np.arange(9),
    average="macro",
    zero_division=0
)

print("\n======================")
print(f"LightGBM OOF Macro-F1: {final_f1:.4f}")
print("======================")

assert oof_lgb.shape == (3278, 9)
assert test_lgb.shape == (296, 9)

np.save("oof_lgb.npy", oof_lgb)
np.save("test_lgb.npy", test_lgb)

print("Fisierele .npy au fost salvate!")



=== FOLD 1/5 ===
Macro-F1 fold 1: 0.2569

=== FOLD 2/5 ===
Macro-F1 fold 2: 0.2482

=== FOLD 3/5 ===
Macro-F1 fold 3: 0.2511

=== FOLD 4/5 ===
Macro-F1 fold 4: 0.2629

=== FOLD 5/5 ===
Macro-F1 fold 5: 0.2776

LightGBM OOF Macro-F1: 0.2598
Fisierele .npy au fost salvate!


In [9]:

from sklearn.metrics import classification_report

print("=== LIGHTGBM: RAPORT PE CLASE ===")

print(classification_report(
    y_train,
    oof_lgb.argmax(axis=1),
    labels=np.arange(9),
    target_names=EVASION_CLASSES,
    zero_division=0
))

assert np.isfinite(oof_lgb).all()
assert np.isfinite(test_lgb).all()

assert np.allclose(oof_lgb.sum(axis=1), 1.0)
assert np.allclose(test_lgb.sum(axis=1), 1.0)

print("\nProbabilitatile sunt valide!")

print("OOF:", np.load("oof_lgb.npy").shape)
print("Test:", np.load("test_lgb.npy").shape)


=== LIGHTGBM: RAPORT PE CLASE ===
                     precision    recall  f1-score   support

           Explicit       0.71      0.39      0.50      1801
           Implicit       0.13      0.18      0.15       276
            General       0.17      0.21      0.19       438
Partial/half-answer       0.01      0.02      0.01        56
            Dodging       0.21      0.38      0.27       286
         Deflection       0.09      0.16      0.12       179
Declining to answer       0.14      0.29      0.18       148
   Claims ignorance       0.30      0.47      0.37        58
      Clarification       0.50      0.58      0.54        36

           accuracy                           0.33      3278
          macro avg       0.25      0.30      0.26      3278
       weighted avg       0.47      0.33      0.37      3278


Probabilitatile sunt valide!
OOF: (3278, 9)
Test: (296, 9)


In [10]:

from sklearn.metrics import f1_score

baseline_oof = np.load(find_file("oof_baseline.npy"))

assert baseline_oof.shape == oof_lgb.shape

for w in [0.25, 0.50, 0.75]:
    combined = (
        w * baseline_oof +
        (1 - w) * oof_lgb
    )

    predictions = combined.argmax(axis=1)

    score = f1_score(
        y_train,
        predictions,
        labels=np.arange(9),
        average="macro",
        zero_division=0
    )

    print(
        f"Baseline weight={w:.2f}, "
        f"LightGBM weight={1-w:.2f}, "
        f"Macro-F1={score:.4f}"
    )


Baseline weight=0.25, LightGBM weight=0.75, Macro-F1=0.2679
Baseline weight=0.50, LightGBM weight=0.50, Macro-F1=0.2857
Baseline weight=0.75, LightGBM weight=0.25, Macro-F1=0.3021


In [11]:
baseline_test = np.load(find_file("test_baseline.npy"))

assert baseline_oof.shape == oof_lgb.shape
assert baseline_test.shape == test_lgb.shape

assert np.isfinite(baseline_oof).all()
assert np.isfinite(baseline_test).all()

assert np.allclose(baseline_oof.sum(axis=1), 1.0)
assert np.allclose(baseline_test.sum(axis=1), 1.0)

print("Baseline OOF:", baseline_oof.shape)
print("Baseline Test:", baseline_test.shape)
print("LightGBM OOF:", oof_lgb.shape)
print("LightGBM Test:", test_lgb.shape)

print("\nVerificarile au trecut!")


Baseline OOF: (3278, 9)
Baseline Test: (296, 9)
LightGBM OOF: (3278, 9)
LightGBM Test: (296, 9)

Verificarile au trecut!


In [12]:

from sklearn.metrics import f1_score

clarity_classes = [
    "Clear Reply",
    "Ambivalent",
    "Clear Non-Reply"
]

EVASION_TO_CLARITY = {
    "Explicit": "Clear Reply",
    "Implicit": "Ambivalent",
    "General": "Ambivalent",
    "Partial/half-answer": "Ambivalent",
    "Dodging": "Ambivalent",
    "Deflection": "Ambivalent",
    "Declining to answer": "Clear Non-Reply",
    "Claims ignorance": "Clear Non-Reply",
    "Clarification": "Clear Non-Reply"
}

combined_oof = 0.75 * baseline_oof + 0.25 * oof_lgb

clarity_probs = np.zeros((len(train), 3))

for j, class_name in enumerate(EVASION_CLASSES):
    parent = EVASION_TO_CLARITY[class_name]
    parent_idx = clarity_classes.index(parent)
    clarity_probs[:, parent_idx] += combined_oof[:, j]

# Predicții pentru Subtask 2
pred_9 = combined_oof.argmax(axis=1)

# Predicții pentru Subtask 1
pred_3 = [
    clarity_classes[i]
    for i in clarity_probs.argmax(axis=1)
]

# Evaluam ambele subtaskuri
f1_evasion = f1_score(
    y_train,
    pred_9,
    labels=np.arange(9),
    average="macro",
    zero_division=0
)

f1_clarity = f1_score(
    train["clarity_label"],
    pred_3,
    average="macro"
)

print("=== ENSEMBLE OOF ===")
print(f"Subtask 1 (Clarity): {f1_clarity:.4f}")
print(f"Subtask 2 (Evasion): {f1_evasion:.4f}")

print(
    f"Scor total estimat pe OOF: "
    f"{50 * f1_clarity + 50 * f1_evasion:.2f}/100"
)


=== ENSEMBLE OOF ===
Subtask 1 (Clarity): 0.2947
Subtask 2 (Evasion): 0.3021
Scor total estimat pe OOF: 29.84/100


In [13]:
assert len(train) == 3278
assert len(test) == 296

assert train["datapointID"].is_unique
assert test["datapointID"].is_unique

assert train["fold"].notna().all()
assert set(train["fold"]) == {0, 1, 2, 3, 4}

print("Exemple in fiecare fold:")
print(train["fold"].value_counts().sort_index())

assert baseline_oof.shape == (3278, 9)
assert oof_lgb.shape == (3278, 9)

assert baseline_test.shape == (296, 9)
assert test_lgb.shape == (296, 9)

print("\nVerificarile de integritate au trecut!")


Exemple in fiecare fold:
fold
0    656
1    656
2    656
3    655
4    655
Name: count, dtype: int64

Verificarile de integritate au trecut!


In [14]:
combined_test = (
    0.75 * baseline_test +
    0.25 * test_lgb
)

pred_indices = combined_test.argmax(axis=1)

pred_9_test = [
    EVASION_CLASSES[i]
    for i in pred_indices
]

# Subtask 1: mapam strategia catre clasa parinte
pred_3_test = [
    EVASION_TO_CLARITY[c]
    for c in pred_9_test
]

submission = pd.DataFrame({
    "subtaskID": [1] * len(test) + [2] * len(test),
    "datapointID": list(test["datapointID"]) * 2,
    "answer": pred_3_test + pred_9_test
})

assert submission.shape == (592, 3)
assert submission["answer"].notna().all()
assert not submission[
    ["subtaskID", "datapointID"]
].duplicated().any()

submission.to_csv(
    "submission_ensemble.csv",
    index=False
)

print("Submission ensemble actualizat!")
print("\nDistributia Subtask 1:")
print(
    submission[
        submission["subtaskID"] == 1
    ]["answer"].value_counts()
)

print("\nDistributia Subtask 2:")
print(
    submission[
        submission["subtaskID"] == 2
    ]["answer"].value_counts()
)


Submission ensemble actualizat!

Distributia Subtask 1:
answer
Clear Reply        134
Ambivalent         124
Clear Non-Reply     38
Name: count, dtype: int64

Distributia Subtask 2:
answer
Explicit               134
General                 52
Dodging                 32
Declining to answer     30
Implicit                20
Deflection              20
Claims ignorance         5
Clarification            3
Name: count, dtype: int64


In [15]:
print("Dimensiune:", submission.shape)

print("\nRanduri pentru fiecare subtask:")
print(submission["subtaskID"].value_counts())

print("\nDistributia predictiilor:")
for task_id in [1, 2]:
    print(f"\nSubtask {task_id}:")
    print(
        submission.loc[
            submission["subtaskID"] == task_id,
            "answer"
        ].value_counts()
    )

for task_id, valid_labels in [
    (1, clarity_classes),
    (2, EVASION_CLASSES)
]:
    part = submission[submission["subtaskID"] == task_id]

    assert len(part) == len(test)
    assert set(part["datapointID"]) == set(test["datapointID"])
    assert set(part["answer"]).issubset(set(valid_labels))

assert submission.shape == (592, 3)
assert submission.columns.tolist() == [
    "subtaskID", "datapointID", "answer"
]
assert not submission["answer"].isna().any()
assert not submission[["subtaskID", "datapointID"]].duplicated().any()

print("\nSubmission verificat cu succes!")


Dimensiune: (592, 3)

Randuri pentru fiecare subtask:
subtaskID
1    296
2    296
Name: count, dtype: int64

Distributia predictiilor:

Subtask 1:
answer
Clear Reply        134
Ambivalent         124
Clear Non-Reply     38
Name: count, dtype: int64

Subtask 2:
answer
Explicit               134
General                 52
Dodging                 32
Declining to answer     30
Implicit                20
Deflection              20
Claims ignorance         5
Clarification            3
Name: count, dtype: int64

Submission verificat cu succes!


In [16]:

# Metoda A: agregarea probabilitatilor
pred_3_sum = [
    clarity_classes[i]
    for i in clarity_probs.argmax(axis=1)
]

# Metoda B: transformarea clasei prezise
pred_9_oof = [
    EVASION_CLASSES[i]
    for i in combined_oof.argmax(axis=1)
]

pred_3_mapping = [
    EVASION_TO_CLARITY[c]
    for c in pred_9_oof
]

for name, predictions in [
    ("Probability aggregation", pred_3_sum),
    ("Class mapping", pred_3_mapping)
]:
    score = f1_score(
        train["clarity_label"],
        predictions,
        average="macro"
    )

    print(f"\n{name}")
    print(f"Macro-F1: {score:.4f}")
    print(pd.Series(predictions).value_counts())



Probability aggregation
Macro-F1: 0.2947
Ambivalent         3120
Clear Reply          94
Clear Non-Reply      64
Name: count, dtype: int64

Class mapping
Macro-F1: 0.5095
Clear Reply        1475
Ambivalent         1412
Clear Non-Reply     391
Name: count, dtype: int64


In [17]:
X_train.to_csv("train_features.csv", index=False)
X_test.to_csv("test_features.csv", index=False)

# Salvam identificatorii in ordinea predictiilor
train["datapointID"].to_csv(
    "train_prediction_ids.csv",
    index=False
)

test["datapointID"].to_csv(
    "test_prediction_ids.csv",
    index=False
)

np.save("oof_ensemble.npy", combined_oof)

combined_test = 0.75 * baseline_test + 0.25 * test_lgb
np.save("test_ensemble.npy", combined_test)

print("Fisierele pentru integrare au fost salvate!")
print("Train features:", X_train.shape)
print("Test features:", X_test.shape)
print("OOF ensemble:", combined_oof.shape)
print("Test ensemble:", combined_test.shape)


Fisierele pentru integrare au fost salvate!
Train features: (3278, 18)
Test features: (296, 18)
OOF ensemble: (3278, 9)
Test ensemble: (296, 9)


In [18]:
print(find_file("waicc_ensemble_reproducible.py"))

/kaggle/input/datasets/biancanan/waicc-ensemble-script/waicc_ensemble_reproducible.py


In [19]:
import subprocess

script_path = find_file("waicc_ensemble_reproducible.py")

result = subprocess.run(
    ["python", script_path],
    capture_output=True,
    text=True
)

print("=== OUTPUT ===")
print(result.stdout)

print("=== ERORI / AVERTISMENTE ===")
print(result.stderr)

print("Return code:", result.returncode)


=== OUTPUT ===
OOF: Clarity=0.5095, Evasion=0.3021, Combined=40.58/100
Wrote submission_ensemble_reproduced.csv

=== ERORI / AVERTISMENTE ===

Return code: 0


In [20]:
import glob
import os

print("=== FISIERE CSV DISPONIBILE ===")

for path in glob.glob("/kaggle/input/**/*.csv", recursive=True):
    print(path)

print("\n=== FISIERE DIN WORKING ===")

for path in glob.glob("/kaggle/working/*.csv"):
    print(path)


=== FISIERE CSV DISPONIBILE ===
/kaggle/input/datasets/biancanan/date-task2/test_context.csv
/kaggle/input/datasets/biancanan/date-task2/train_context.csv
/kaggle/input/datasets/biancanan/waicc-26/sample_output.csv
/kaggle/input/datasets/biancanan/waicc-26/test_data.csv
/kaggle/input/datasets/biancanan/waicc-26/train_data.csv

=== FISIERE DIN WORKING ===
/kaggle/working/test_features.csv
/kaggle/working/test_prediction_ids.csv
/kaggle/working/train_prediction_ids.csv
/kaggle/working/submission_ensemble_reproduced.csv
/kaggle/working/submission_ensemble.csv
/kaggle/working/train_features.csv


In [21]:
print("Train:", find_file("train_data.csv"))
print("Test:", find_file("test_data.csv"))
print("Sample:", find_file("sample_output.csv"))

print("\nToate fisierele originale sunt disponibile!")


Train: /kaggle/input/datasets/biancanan/waicc-26/train_data.csv
Test: /kaggle/input/datasets/biancanan/waicc-26/test_data.csv
Sample: /kaggle/input/datasets/biancanan/waicc-26/sample_output.csv

Toate fisierele originale sunt disponibile!


In [22]:
import pandas as pd

original = pd.read_csv(
    "/kaggle/working/submission_ensemble.csv"
)

reproduced = pd.read_csv(
    "/kaggle/working/submission_ensemble_reproduced.csv"
)

keys = ["subtaskID", "datapointID"]

comparison = original.merge(
    reproduced,
    on=keys,
    suffixes=("_original", "_reproduced"),
    validate="one_to_one"
)

assert len(original) == 592
assert len(reproduced) == 592
assert len(comparison) == 592

differences = (
    comparison["answer_original"] !=
    comparison["answer_reproduced"]
)

print("Predictii identice:", (~differences).sum())
print("Predictii diferite:", differences.sum())

display(comparison.loc[differences].head(10))


Predictii identice: 592
Predictii diferite: 0


,subtaskID,datapointID,answer_original,answer_reproduced
